# Advanced Farmer-Income Model

This is my final pipeline for predicting a farmer's total income (scored on MAPE), on a single 85/15 split.

The whole idea, short version:
- I don't predict total income straight up. I split it: `agri = total - non_agri`, predict the agri part in log space, then add the non-agri income back at the end - it's a known number, so there's no point making the model guess it.
- The biggest thing I changed from my earlier pipeline: I **keep `Non_Agriculture_Income` as a feature**. It correlates 0.935 with total income and earlier I was throwing it away - putting it back is what took me from ~21% down to ~18.5% MAPE.
- On top of that: leakage-free target encodings for district/village/mandi/state, year-trend slopes plus the latest-year level, a couple of land interactions, and I leave NaNs alone so the trees handle them.
- Model is a LightGBM + XGBoost + CatBoost ensemble on the MAE objective, with a small bias multiplier `k` tuned on train.

Final result: CV MAPE ~18.5%. The full write-up is in `Pipeline.md` and `Project_Report.md`.

## 0. Imports

Just pulling in what I need - pandas/numpy for the data wrangling, the three boosting libraries (they're what actually works on this tabular data), and matplotlib for the residual plot at the end.

In [1]:
import numpy as np, pandas as pd, re, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, KFold
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostRegressor
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

## 1. Config and the MAPE metric

Seed, file name, and the target column up top so everything lives in one place. I define `mape` here too because that's the metric I'm judging everything on - note it skips rows where the true value is 0 (can't divide by zero), and since it's a relative error the low-income farmers end up carrying most of the weight.

In [2]:
RND   = 42
XLSX  = "LTF Challenge data with dictionary.xlsx"
TARGET = "Target_Variable/Total Income"

def mape(y, p):
    """Mean Absolute Percentage Error (%) - the competition metric."""
    y, p = np.asarray(y, float), np.asarray(p, float); m = y != 0
    return np.mean(np.abs(y[m] - p[m]) / y[m]) * 100

## 2. Leakage-free target encoding

This is how I turn a high-cardinality location (district, village, ...) into a usable number: the smoothed mean agri income for that group. The trap is leakage - if I just take the group mean, a row ends up seeing its own income. So:
- train rows get **out-of-fold** values (a row's encoding comes from the *other* folds, never its own),
- val/test rows get the **full-train** map,
- and `prior_map` lets me smooth a group toward its parent (village toward its district) so tiny villages don't go wild. `m` is the smoothing strength - higher means lean more on the prior.

In [3]:
def te_oof(key_tr, y_tr, key_va, gm, prior_map=None, m=20, n_splits=5):
    def smooth(s, idx):
        pr = gm if prior_map is None else idx.to_series().map(prior_map).fillna(gm)
        return (s["mean"] * s["count"] + pr * m) / (s["count"] + m)
    oof = pd.Series(index=key_tr.index, dtype=float)
    for tr, va in KFold(n_splits, shuffle=True, random_state=0).split(key_tr):
        s = y_tr.iloc[tr].groupby(key_tr.iloc[tr]).agg(["mean", "count"])
        sm = smooth(s, s.index)
        oof.iloc[va] = key_tr.iloc[va].map(sm).fillna(gm)
    s = y_tr.groupby(key_tr).agg(["mean", "count"])
    full = smooth(s, s.index)
    return oof, key_va.map(full).fillna(gm), full

## 3. Per-row feature engineering

Everything here is computed row by row and never touches the target, so it's safe on any split. What I'm building:
- loan amount x number of loans (and I keep the raw ones too),
- `has_river` / `has_water` flags from the 2022 water-body text,
- temperature: the cells are "min/max" strings, so I parse them and keep the per-season **max** - min and max are basically collinear, no point keeping both,
- for every multi-year metric I keep **both** the slope (the trend) and the latest-year value (the current state) - I think the current level matters more than the trend for predicting current income,
- rabi rainfall slope,
- land transforms (log land + a missing-flag),
- one-hot for the low-cardinality categoricals.

Then I drop the raw text/geo/year columns I've already summarised, but I keep the location keys (district/village/mandi/state) because I encode those in the next step.

In [4]:
def base_features(df):
    df = df.set_index("FarmerID").copy()
    df.columns = df.columns.str.replace(r"\s+", " ", regex=True).str.strip()

    df["Total_outstanding_loan"] = df["Avg_Disbursement_Amount_Bureau"] * df["No_of_Active_Loan_In_Bureau"]

    wb_col = [c for c in df.columns if "Kharif" in c and "Type of water bodies" in c and "2022" in c]
    if wb_col:
        wb = df[wb_col[0]].fillna("[None]").astype(str)
        df["has_river"] = wb.str.contains("river", case=False).astype(int)
        df["has_water"] = wb.str.contains("water", case=False).astype(int)

    tcols = [c for c in df.columns if "Ambient temperature" in c]
    if tcols:
        mx = {c: pd.to_numeric(df[c].astype(str).str.split("/", expand=True).get(1), errors="coerce") for c in tcols}
        MX = pd.DataFrame(mx)
        df["kharif_temp_max"] = MX[[c for c in tcols if c.startswith("K")]].mean(1)
        df["rabi_temp_max"]   = MX[[c for c in tcols if c.startswith("R")]].mean(1)

    groups = {}
    for c in df.columns:
        if re.search(r"20(20|21|22)", c) and df[c].dtype != "object":
            k = re.sub(r"\s*20(20|21|22)\s*", " ", c).strip(); groups.setdefault(k, []).append(c)
    for k, cols in {k: sorted(v) for k, v in groups.items() if len(v) > 1}.items():
        yrs = np.array([int(re.search(r"20(20|21|22)", c).group()) for c in cols], float); w = yrs - yrs.mean()
        sub = df[cols].astype(float)
        df[k + "_slope"]  = (sub.values * w).sum(1) / (w ** 2).sum()
        df[k + "_latest"] = df[cols[int(np.argmax(yrs))]].astype(float).values
        df = df.drop(columns=cols)

    rr = [c for c in df.columns if "Seasonal Average Rainfall" in c and c.startswith("R")]
    if len(rr) >= 2:
        yrs = np.array([2020, 2021, 2022], float)[: len(rr)]; w = yrs - yrs.mean()
        df["rabi_rain_slope"] = (df[sorted(rr)].astype(float).values * w).sum(1) / (w ** 2).sum()

    if "Total_Land_For_Agriculture" in df.columns:
        df["log_land"] = np.log1p(df["Total_Land_For_Agriculture"].clip(lower=0))
        df["Total_Land_missing"] = df["Total_Land_For_Agriculture"].isna().astype(int)

    ohe = [c for c in ["SEX", "REGION", "MARITAL_STATUS",
                       "R022-Village category based on Agri parameters (Good, Average, Poor)"] if c in df.columns]
    df = pd.get_dummies(df, columns=ohe, dummy_na=False)

    drop_like = ["Type of soil", "Agro Ecological", "Type of water bodies", "Ambient temperature",
                 "Seasonal Average Rainfall", "Nearest Mandi Name", "Total Geographical Area",
                 "category based on socio-economic", "Address type", "Ownership", "Location", "Zipcode"]
    df = df.drop(columns=[c for c in df.columns if any(s in c for s in drop_like)], errors="ignore")
    return df

## 4. Master builder

This is where it all comes together: the per-row features + the leakage-free target encodings (fit on train only) + the land interactions. District is smoothed toward the global mean, village toward its own district (that's the hierarchy), mandi and state toward global. I add `land x district_te` and `land x village_te` because income is roughly scale x locality. At the end I keep only the numeric columns, line train and val up to the exact same columns, and clean the names so LightGBM/XGBoost don't choke on the special characters. The thing I actually model is `agri = total - non_agri` (clipped at 0); `non_agri` stays in as a feature.

In [5]:
def build(raw_tr, raw_va):
    keys = ["State", "DISTRICT", "VILLAGE", "K022-Nearest Mandi Name"]
    ktr = {k: raw_tr.set_index("FarmerID")[k] for k in keys if k in raw_tr.columns}
    kva = {k: raw_va.set_index("FarmerID")[k] for k in keys if k in raw_va.columns}

    nai_tr = raw_tr.set_index("FarmerID")["Non_Agriculture_Income"]
    nai_va = raw_va.set_index("FarmerID")["Non_Agriculture_Income"]
    tot_tr = raw_tr.set_index("FarmerID")[TARGET]
    tot_va = raw_va.set_index("FarmerID")[TARGET]
    agri_tr = (tot_tr - nai_tr).clip(lower=0)

    Xtr = base_features(raw_tr.drop(columns=[TARGET]))
    Xva = base_features(raw_va.drop(columns=[c for c in [TARGET] if c in raw_va.columns]))

    gm = agri_tr.mean()
    d_oof, d_va, d_full = te_oof(ktr["DISTRICT"], agri_tr, kva["DISTRICT"], gm)
    vil_dist = pd.DataFrame({"v": ktr["VILLAGE"].values, "d": ktr["DISTRICT"].values}).groupby("v")["d"].first()
    v_prior = vil_dist.map(d_full)
    v_oof, v_va, _ = te_oof(ktr["VILLAGE"], agri_tr, kva["VILLAGE"], gm, prior_map=v_prior, m=10)
    m_oof, m_va, _ = te_oof(ktr["K022-Nearest Mandi Name"], agri_tr, kva["K022-Nearest Mandi Name"], gm)
    s_oof, s_va, _ = te_oof(ktr["State"], agri_tr, kva["State"], gm)
    for name, tr_enc, va_enc in [("DISTRICT_te", d_oof, d_va), ("VILLAGE_te", v_oof, v_va),
                                 ("MANDI_te", m_oof, m_va), ("STATE_te", s_oof, s_va)]:
        Xtr[name] = tr_enc.reindex(Xtr.index); Xva[name] = va_enc.reindex(Xva.index)

    for X in (Xtr, Xva):
        if "Total_Land_For_Agriculture" in X and "DISTRICT_te" in X:
            X["land_x_dist"] = X["Total_Land_For_Agriculture"] * X["DISTRICT_te"]
            X["land_x_vill"] = X["Total_Land_For_Agriculture"] * X["VILLAGE_te"]

    Xtr = Xtr.select_dtypes(include="number"); Xva = Xva.select_dtypes(include="number")
    Xva = Xva.reindex(columns=Xtr.columns, fill_value=0)
    safe = {c: re.sub(r"[^0-9A-Za-z_]+", "_", str(c)).strip("_") for c in Xtr.columns}
    Xtr = Xtr.rename(columns=safe); Xva = Xva.rename(columns=safe)
    return Xtr, Xva, agri_tr, tot_tr, tot_va, nai_tr, nai_va

## 5. Load, split 85/15, build features

Read the train sheet, do the 85/15 split, and run both halves through `build`. `ytr` is `log1p(agri income)` - that's what the models are actually trained on.

In [6]:
raw = pd.read_excel(XLSX, sheet_name="TrainData")
raw_tr, raw_va = train_test_split(raw, test_size=0.15, random_state=RND)
Xtr, Xva, agri_tr, tot_tr, tot_va, nai_tr, nai_va = build(raw_tr, raw_va)
ytr = np.log1p(agri_tr.values)
print(f"features={Xtr.shape[1]}  train={len(Xtr)}  cv={len(Xva)}")

features=62  train=40774  cv=7196


## 6. Train the ensemble + bias correction

Three boosting models, all on MAE - I want the **median**, not the mean, because that's what MAPE rewards - averaged together in income space. Then I tune a single multiplicative `k` on the train set to nudge the overall level, and apply it to CV. I print MAPE and accuracy (100 - MAPE) for both train and CV; CV is the number that actually matters.

In [7]:
models = {
    "lgb": lgb.LGBMRegressor(objective="regression_l1", n_estimators=3000, learning_rate=0.015,
                             num_leaves=255, min_child_samples=40, subsample=0.85, subsample_freq=1,
                             colsample_bytree=0.7, reg_lambda=3.0, random_state=RND, n_jobs=-1, verbose=-1),
    "xgb": xgb.XGBRegressor(objective="reg:absoluteerror", n_estimators=2500, learning_rate=0.015,
                            max_depth=10, min_child_weight=10, subsample=0.85, colsample_bytree=0.7,
                            reg_lambda=3.0, gamma=1.0, tree_method="hist", random_state=RND, n_jobs=-1),
    "cat": CatBoostRegressor(loss_function="MAE", iterations=2500, depth=9, learning_rate=0.02,
                             l2_leaf_reg=5.0, random_state=RND, verbose=0),
}
atr = np.zeros(len(Xtr)); ava = np.zeros(len(Xva))
for name, mdl in models.items():
    mdl.fit(Xtr, ytr)
    atr += np.expm1(mdl.predict(Xtr)); ava += np.expm1(mdl.predict(Xva))
atr /= len(models); ava /= len(models)                       # ensemble agri-income predictions

ks = np.linspace(0.80, 1.20, 81)
k  = ks[np.argmin([mape(tot_tr.values, atr * kk + nai_tr.values) for kk in ks])]
pred_tr = atr * k + nai_tr.values
pred_va = ava * k + nai_va.values

from sklearn.metrics import r2_score
def report_row(name, y, p):
    m = mape(y, p)
    print(f"  {name:<6} MAPE {m:6.2f}%    Accuracy {100-m:6.2f}%    R2 {r2_score(y, p):.3f}")
print("metric table (reconstructed TOTAL income):")
report_row("train", tot_tr.values, pred_tr)
report_row("CV",    tot_va.values, pred_va)
print(f"  bias k = {k:.3f}")

metric table (reconstructed TOTAL income):
  train  MAPE  12.69%    Accuracy  87.31%    R2 0.945
  CV     MAPE  18.48%    Accuracy  81.52%    R2 0.941
  bias k = 0.995
